# Delhi Air Quality and Voting

In this exercise, we will continue our exploration of Delhi air quality from our previous exercise. But we will also be adding election data to answer the question: are people experiencing worse air quality more likely to vote *against* the incumbent political party in Delhi?

To answer this question, we will bring State Assembly boundaries provided by the [Data{Meet} Community Maps Project](https://projects.datameet.org/maps/assembly-constituencies/#attribute) and Assembly-level electoral returns provided by [Ashoka University's Trivedi Centre for Political Data](https://tcpd.ashoka.edu.in/lok-dhaba/).

> A note to Indian users or users likely to travel to India: a colleague tells me that this site's shapefiles are scraped from the Indian Department of Rural Development, and so reflect Government approved boundaries for Jammu and Kashmir, and so *should* be safer than those from some internal sources to use and publish. Note that in this exercise we will only use data on Delhi.


## Gradescope Autograding

Please follow [all standard guidance](https://www.practicaldatascience.org/ids720_specific/autograder_guidelines.html) for submitting this assignment to the Gradescope autograder, including storing your solutions in a dictionary called `results` and ensuring your notebook runs from the start to completion without any errors.

For this assignment, please name your file `exercise_projections.ipynb` before uploading.

You can check that you have answers for all questions in your `results` dictionary with this code:

```python
assert set(results.keys()) == {
    "ex6_world_crs_str",
    "ex9_delhi_mean_so2",
    "ex9_delhi_crs_str",
    "ex12_distance",
    "ex12_nearest_name",
    "ex13_distance_units",
    "ex14_distance",
    "ex14_nearest_name",
    "ex15_change",
    "ex17_utmzone",
    "ex18_distance",
    "ex18_nearest_name",
    "ex18_units",
}
```


### Submission Limits

Please remember that you are **only allowed THREE submissions to the autograder.** Your last submission (if you submit 3 or fewer times), or your third submission (if you submit more than 3 times) will determine your grade Submissions that error out will **not** count against this total.

### Exercise 1

Please load the air pollution data [from this URL](https://github.com/nickeubank/MIDS_Data/blob/master/air_pollution/other/delhi_air_krig.geojson). What you will get is a GeoDataFrame, but if you plot it, you will see it looks a lot like the raster we created by kriging in the last exercise. That's because... it is! Or rather, this is a *vectorized* version of that data — we took that grid, created a polygon for each grid square, and put it in a GeoDataFrame as its own row with the associated value. 

This isn't something we'll do much, but since we haven't worked with raster data yet, this way we can have that nice interpolated grid of values for all of Delhi while still only working with geopandas. 

So load the data and plot it.

### Exercise 2

Now download Assembly District boundaries from this [URL](https://github.com/nickeubank/MIDS_Data/blob/master/delhi_elections/delhi_assembly_districts.geojson). Again, these are from [Data{Meet} Community Maps Project](https://projects.datameet.org/maps/assembly-constituencies/#attribute), but this is already filtered down to Delhi.

For Indiaphiles: [These appear to have been scraped in 2016](https://github.com/datameet/maps/issues/86), and so at least for Delhi are Post-Delimitation boundaries.

### Exercise 3

As you look at the Assembly District boundaries, you will notice a pronounced lack of electoral data! This is actually not all that uncommon — many times spatial boundaries and the tabular data you may wish to join with the spatial boundaries are provided separately.

Download the 2020 Delhi Assembly electoral returns [from here](https://github.com/nickeubank/MIDS_Data/blob/master/delhi_elections/2020_delhi_assembly_returns.csv). This data comes from [this site](https://tcpd.ashoka.edu.in/lok-dhaba/), and you can find the codebook for the [data here](https://lokdhaba.ashoka.edu.in/static/media/2022Feb12LokDhabaCodebook.21040cf7.pdf). 

Before we merge this data with our Assembly District boundaries, it can be helpful to clean the data so we have only one row in our tabular election-return data per Assembly District.

Since not everyone here is a political scientist: each row of this data corresponds to a District-Candidate pair, and has data on the votes earned by a given candidate in the given district. As such, there are many rows per Assembly District (since many candidates run in each District).

Since we're just interested in the vote share of the incumbent party in Delhi (the Aam Aadmi Party (the "common man"  party), denoted `"AAAP"` in the `Party` column). This can be accomplished through subsetting and the use of the `Votes` and `Valid_Votes` columns (see the codebook if needed).

Calculate the average AAAP vote share across Districts and store your answer in `"ex3_aaap_avg"`.

### Exercise 4

Now that you only have one record per Assembly District, let's merge our election returns with our District boundaries.

**Note:** to ensure you get back a GeoDataFrame, it is recommended you use the `.merge` syntax with the GeoDataFrame coming first:

```python
my_geodataframe.merge(my_pandas_dataframe, on="common_var")
```

You can also get away with `pd.merge` if the GeoDataFrame is in the left position, apparently, but the above syntax is recommended. If you don't do this, the merge will work but you'll end up with a pandas DataFrame rather than a GeoDataFrame.

So now merge your data. Store the kind of merge you are doing as a string (`"1:1"`, `"1:m"`, `"m:1"`, or `"m:m"`) under `"ex4_merge_type"`. (forget about merge validation? [Re-read here](https://www.practicaldatascience.org/notebooks/class_3/week_4/15.3_validating_a_merge.html)!)

### Exercise 5

As a first, simple exercise, plot both Assembly Districts and air quality monitor air pollution on the same plot. Color air quality monitor markers according to SO2 levels and Assembly Districts according to AAAP vote share.

Note: if this doesn't work at first, you probably forgot something important! :) 

## Correlating Pollution and Incumbent Party Support

In our next few exercises, we are going to calculate the average air pollution in across each District. We will then correlate that average with the District's AAAP vote share to see if AAAP vote share is lower in Districts with worse air pollution. 

First, though, a few caveats are in order:

- This is not a good empirical strategy for estimating the causal effect of air quality on support for the incumbent party. Air pollution tends to be worse in Delhi's city center, and the type of people who live in the city center are not the same as the type of people who live on the city's outskirts. Given that, our correlation may be telling us more about the type of people who support the AAAP than the effect of air pollution. 
    - If this were a serious research project, a simple strategy that would improve on this immensely would be to collect air pollution data and election returns over a longer period of time and measure how *changes* in air pollution correlate with *changes* in incumbent support at the District level. But this is just a class exercise, so... 🤷‍♂️
- These Air Quality stations are also not the best data available on air pollution averaged over a longer period of time — that would come from satellite data as we'll see in the coming weeks. These type of stations are primarily valuable for measuring pollution in real time.

But with those caveats out of the way, let's do some analyses!


### Exercise 6

We now have two GeoDataFrames full of Polygon shapes we want to join. This is actually a slightly tricky thing to do. When one polygon lies fully within another, of course, we know they should join, and if two polygons don't touch at all, they should not. But what about partially overlapping polygons?

The geopandas `sjoin` method provides a few options, which you can read about in the [Merging Data](https://geopandas.org/en/stable/docs/user_guide/mergingdata.html#binary-predicate-joins) documentation under the `predicates` section. In particular, one can join records that have any of the following relationships to one another:

- intersects
- contains
- within
- touches
- crosses
- overlaps

Let's start with the simplest option: joining our constituency polygons to the pollution polygons that are fully within each constituency. This should be a `left` join.

**Note** to ensure we're all working with the same data, make sure you're using `UTM zone 43N` as your CRS. You probably switched to it in Exercise 5, but if you projected both datasets into Lat-Long (epsg 4326), well... shame on you. :) Use UTM zone 43N.

When you are done, store the number of rows in your resulting dataset in `"ex6_nrows"`.

### Exercise 7

How many constituencies have no SO2 pollution data? Store your answer in `"ex7_no_pollution"`. Why do you think this happened? Do they not have air? Answer in markdown.

### Exercise 8

How many pollution polygons did you have in your raw data and how many ended up in the merged data (i.e., how many rows of the merged data have SO2_predicted values)? Store your answers in `"ex8_n_in_raw"` and `"ex8_n_in_merged"`. Why did this happen? Answer in Markdown.

### Exercise 9

OK, we've now see the issues that arise within `within`. Now let's try `intersects`. Before we try, can you predict what will happen?

Store the number of Districts with no pollution data in `"ex9_no_pollution"` and the number of pollution records (rows with pollution data) in `"ex9_n_in_merged"`. How does `"ex9_n_in_merged"` compare to `"ex8_n_in_raw"` and `"ex8_n_in_merged"`?

### Exercise 10

Now that you've seen the two extremes of polygon merging, let's look for something in the middle. 

First, we'll take try the "quick and dirty" strategy.

Take your pollution data, and replace your polygons with centroids. A [centroid](https://en.wikipedia.org/wiki/Centroid) is a single point in the "middle" of a polygon, where "middle" is defined as the point where you could balance the polygon on a pin if you cut it out of a uniform material. 

Make sure you both construct your centroids *and* make them the active geometry before merging!

Then merge your Districts with all the centroids they contain. 

Store the number of districts with no polygon data in `"ex10_no_pollution"` and the number of rows with pollution data in `"ex10_n_in_merged"`. Note that `"ex10_n_in_merged"` may seem a little lower than you'd expect, but if you look at your plot in Exercise 5 you can probably tell why.

### Exercise 11

And now for the final, complicated operation. 

Basically, we are going to say that the air pollution in each constituency is an *area-weighed average* of the pollution in each intersecting polygon. That means that all polygons that are fully within a district will get equal weight, but a polygon that is only half-in a district will only get half-weight.

To do so, we will first have to *overlay* our two polygon layers to create a new GeoDataFrame. In particular, we will need to `intersect` our polygon layers. If you don't remember set operations, [please review them now](https://geopandas.org/en/stable/docs/user_guide/set_operations.html)!

Union your two layers. Store the resulting number of polygons in `"ex11_num_intersected_polygons"`.

### Exercise 12

Now in order to take an area-weighted average, we will need each polygon's area! Create a new column with each polygon's area using `.area`.

### Exercise 13

And now we need to do a good old-fashioned tabular `groupby`. Basically, for each District, we want to calculate:

$$\sum_{i \in D} \frac{area_i}{(\sum_{i \in D} area_i)} * \text{SO2\_predicted}_i $$

Where $i$ is an index of all polygons in District D.

This weighted average looks scary at first, but it isn't bad, I promise. 

If $area_i = 1$ for all $i$, for example, then we'd just be adding up all the SO2 predicted values and dividing by N.

In Python-speak (instead of math-speak), for each district we want to do:

```python
district_size = polygons_in_district["area"].sum()

numerator = 0
for row in polygons_in_district.itertuples():
    numerator = row.area * row.SO2_predicted

district_area_weighted_SO2 = numerator / district_size
```

Now as with most things data science, I wouldn't do it in a loop, but if you find that helpful there it is!